# Flickr30k Dataset Analysis with CLIP

This notebook demonstrates training and evaluation of the CLIP model on the Flickr30k dataset, including data preprocessing, model training, and performance analysis.

In [1]:
import torch

print("MPS available:", torch.backends.mps.is_available())
print("Device:", "mps" if torch.backends.mps.is_available() else "cpu")


MPS available: True
Device: mps


### Cell 2: Import dependencies

`import sys`


In [2]:
import sys
!{sys.executable} -m pip uninstall numpy -y
!{sys.executable} -m pip install numpy==1.26.4



Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp310-cp310-macosx_11_0_arm64.whl (14.0 MB)

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Importing Libraries

This cell imports the necessary libraries for the notebook, such as PyTorch, NumPy, and other dependencies required for data processing and model training.

In [3]:
import numpy as np
print(np.__version__)


1.26.4


In [4]:
pip install "datasets>=2.10,<2.18"


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 

## Data Loading

In [5]:
from datasets import load_dataset

dataset = load_dataset("nlphuji/flickr30k")

dataset


/Users/ssingodia/Desktop/Project-3/clipenv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    test: Dataset({
        features: ['image', 'caption', 'sentids', 'split', 'img_id', 'filename'],
        num_rows: 31014
    })
})

In [6]:
dataset["test"][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500>,
 'caption': ['Two young guys with shaggy hair look at their hands while hanging out in the yard.',
  'Two young, White males are outside near many bushes.',
  'Two men in green shirts are standing in a yard.',
  'A man in a blue shirt standing in a garden.',
  'Two friends enjoy time spent together.'],
 'sentids': ['0', '1', '2', '3', '4'],
 'split': 'train',
 'img_id': '0',
 'filename': '1000092795.jpg'}

In [7]:
full_dataset = dataset["test"]
print(len(full_dataset))


31014


### Splits the Dataset into Train / Val / Test
This cell shuffles the full dataset with a fixed seed for reproducibility, then splits it into training (80%), validation (10%), and test (10%) subsets. The sizes of each split are printed to confirm the split proportions.

In [8]:
full_dataset = full_dataset.shuffle(seed=42)

train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset = full_dataset.select(range(train_size))
val_dataset = full_dataset.select(range(train_size, train_size + val_size))
test_dataset = full_dataset.select(range(train_size + val_size, len(full_dataset)))

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))
   

Train: 24811
Val: 3101
Test: 3102


In [13]:
# train_pairs = []

# for item in train_dataset:
#     image = item["image"]
#     captions = item["caption"]
    
#     for caption in captions:
#         train_pairs.append((image, caption))

# len(train_pairs)


# ==========================================
# CONTROLLED TRAIN DATA (40K WITH 25K UNIQUE IMAGES)
# ==========================================


### Builds Controlled Training Pairs (One Caption Per Image)
This cell constructs a controlled training dataset by mapping each unique image to all of its captions, then selecting exactly one caption per image to form `unique_image_pairs`. The remaining captions are stored in `remaining_pairs` for optional augmentation. The number of unique images and remaining pairs are printed.

In [9]:
from collections import defaultdict
import random

image_to_captions = defaultdict(list)

# Step 1: Build mapping
for item in train_dataset:
    image = item["image"]
    captions = item["caption"]
    
    for caption in captions:
        image_to_captions[id(image)].append((image, caption))

print("Unique images:", len(image_to_captions))


# Step 2: Select 1 caption per image and store rest separately
unique_image_pairs = []
remaining_pairs = []

for image_id, captions_list in image_to_captions.items():
    
    # Randomly choose one caption for uniqueness
    chosen_pair = random.choice(captions_list)
    unique_image_pairs.append(chosen_pair)
    
    # Add the remaining captions directly (no comparison needed)
    for pair in captions_list:
        if pair != chosen_pair:
            remaining_pairs.append(pair)

print("Unique pairs:", len(unique_image_pairs))
print("Remaining pairs:", len(remaining_pairs))

Unique images: 24811
Unique pairs: 24811
Remaining pairs: 99234


### Samples Additional Caption Pairs
This cell randomly selects 15,189 additional image-caption pairs from the remaining pool using a fixed seed. Combined with the unique image pairs, this brings the total training set to approximately 40K pairs.

In [10]:
# Step 4: Randomly select 15K additional pairs

random.seed(42)

additional_pairs = random.sample(remaining_pairs, 15189)

print("Additional pairs selected:", len(additional_pairs))

Additional pairs selected: 15189


In [11]:
# Step 5: Combine to make final 40K training dataset

final_train_pairs = unique_image_pairs + additional_pairs

random.shuffle(final_train_pairs)

print("Final training dataset size:", len(final_train_pairs))

Final training dataset size: 40000


In [17]:
# import random

# train_pairs = random.sample(train_pairs, 30000)

# # train_pairs = train_pairs[:30000]

In [12]:
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


/Users/ssingodia/Desktop/Project-3/clipenv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


# leraned temp, def.

### Sets the Compute Device
This cell detects and assigns the appropriate compute device — CUDA GPU if available, otherwise CPU — to the `device` variable used throughout the notebook for tensor and model placement.

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Initializes the Learnable Temperature (`logit_scale`)
This cell creates a learnable `logit_scale` parameter initialized to `log(1/0.07)`, which corresponds to the default CLIP temperature of 0.07. It is moved to the active device and will be jointly optimized with the model parameters during training.

In [14]:
import torch
import torch.nn as nn
import numpy as np

logit_scale = nn.Parameter(torch.tensor(np.log(1/0.07), dtype=torch.float32))
logit_scale = logit_scale.to(device)

### Cleans and Validates Training Pairs
This cell iterates over `final_train_pairs`, filtering out any entries with `None` images, converting all images to RGB format, and ensuring captions are valid strings. The cleaned pairs are stored in `clean_pairs`, and the total count is printed.

In [ ]:
# clean_pairs = []

# for img, cap in final_train_pairs:
# # for img, cap in train_pairs:
#     try:
#         if img is None:
#             continue
            
#         # ensure PIL image RGB
#         img = img.convert("RGB")
        
#         # ensure caption string
#         cap = str(cap)
        
#         clean_pairs.append((img, cap))
        
#     except:
#         continue

# len(clean_pairs)


In [ ]:
# img = train_pairs[0][0]
# print(img)
# print(type(img))
# print(img.mode if hasattr(img, "mode") else "no mode")


<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x375 at 0xE345ABA00>
<class 'PIL.JpegImagePlugin.JpegImageFile'>
RGB


In [ ]:
# from transformers import CLIPProcessor

# processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# img, cap = train_pairs[0]

# inputs = processor(
#     images=[img],
#     text=[cap],
#     return_tensors="pt",
#     padding=True,
#     truncation=True
# )

# for k,v in inputs.items():
#     print(k, v.shape)


input_ids torch.Size([1, 10])
attention_mask torch.Size([1, 10])
pixel_values torch.Size([1, 3, 224, 224])


In [ ]:
# batch = train_pairs[:4]

# images = [i[0] for i in batch]
# captions = [i[1] for i in batch]

# inputs = processor(
#     images=images,
#     text=captions,
#     return_tensors="pt",
#     padding=True,
#     truncation=True
# )

# for k,v in inputs.items():
#     print(k, v.shape)


input_ids torch.Size([4, 23])
attention_mask torch.Size([4, 23])
pixel_values torch.Size([4, 3, 224, 224])


In [15]:
from torch.utils.data import Dataset

### Defines the `FlickrDataset` Class
This cell defines a custom PyTorch `Dataset` that wraps a list of image-caption pairs. It implements `__len__` and `__getitem__`, making it compatible with PyTorch's `DataLoader` for efficient batched data loading during training and evaluation.

In [16]:
class FlickrDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        image, caption = self.pairs[idx]
        return image, caption


### Defines the Collate Function
This cell (re)initializes the `CLIPProcessor` and defines `collate_fn`, which unpacks a batch of image-caption pairs and uses the processor to tokenize the text and preprocess the images into padded tensors ready for model input.

In [17]:
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def collate_fn(batch):
    images, captions = zip(*batch)
    
    inputs = processor(
        text=list(captions),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    
    return inputs


In [18]:
from torch.utils.data import DataLoader

### Defines the Collate Function
This cell (re)initializes the `CLIPProcessor` and defines `collate_fn`, which unpacks a batch of image-caption pairs and uses the processor to tokenize the text and preprocess the images into padded tensors ready for model input.

In [19]:
# train_data = FlickrDataset(train_pairs)

# train_loader = DataLoader(
#     train_data,
#     batch_size=32,
#     shuffle=True,   
#     num_workers=0,
#     collate_fn=collate_fn
# )

train_pairs = final_train_pairs

train_data = FlickrDataset(train_pairs)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

### Defines the Collate Function
This cell (re)initializes the `CLIPProcessor` and defines `collate_fn`, which unpacks a batch of image-caption pairs and uses the processor to tokenize the text and preprocess the images into padded tensors ready for model input.

In [20]:
batch = next(iter(train_loader))

for key in batch:
    print(key, batch[key].shape)


input_ids torch.Size([32, 28])
attention_mask torch.Size([32, 28])
pixel_values torch.Size([32, 3, 224, 224])


In [21]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

batch = {k: v.to(device) for k, v in batch.items()}


In [ ]:
# import pickle

# with open("../data/train_pairs.pkl", "wb") as f:
#     pickle.dump(train_pairs, f)

# Phase 4


### Loads the Pretrained CLIP Model for Fine-Tuning
This cell detects the compute device, loads the `openai/clip-vit-base-patch32` CLIP model, moves

In [22]:
import torch
from transformers import CLIPModel

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model = model.to(device)

model.train()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [23]:
batch = next(iter(train_loader))
print(batch.keys())

dict_keys(['input_ids', 'attention_mask', 'pixel_values'])


In [ ]:
# batch = {k: v.to(device) for k, v in batch.items()}

# outputs = model(**batch)

# image_embeds = outputs.image_embeds
# text_embeds = outputs.text_embeds

# print(image_embeds.shape)
# print(text_embeds.shape)

torch.Size([32, 512])
torch.Size([32, 512])


In [ ]:
# import torch.nn.functional as F

# image_embeds = F.normalize(image_embeds, dim=1)
# text_embeds = F.normalize(text_embeds, dim=1)

In [ ]:
# similarity = image_embeds @ text_embeds.T

# print(similarity.shape)

torch.Size([32, 32])


In [ ]:
# labels = torch.arange(similarity.size(0)).to(device)

In [ ]:
# loss_i = F.cross_entropy(similarity, labels)
# loss_t = F.cross_entropy(similarity.T, labels)

# loss = (loss_i + loss_t) / 2

# print(loss)

tensor(1.7177, device='mps:0', grad_fn=<DivBackward0>)


## Optimizer 

### Initializes the AdamW Optimizer
This cell sets up the `AdamW` optimizer with a learning rate of `2e-6`, jointly optimizing all CLIP model parameters along with the learnable `logit_scale` temperature. A commented-out variant shows an earlier layer-wise learning rate strategy that assigned different rates to the vision encoder, text encoder, projection layers, and temperature parameter.

In [24]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-6)


from torch.optim import AdamW

logit_scale = logit_scale.detach().requires_grad_()
optimizer = AdamW(
    list(model.parameters()) + [logit_scale],
    lr=2e-6
)

# from torch.optim import AdamW

# # make sure logit_scale is a leaf parameter
# logit_scale = logit_scale.detach().requires_grad_()

# optimizer = AdamW([
    
#     # Vision encoder (image model) → smaller LR
#     {"params": model.vision_model.parameters(), "lr": 1e-6},
    
#     # Text encoder → higher LR
#     {"params": model.text_model.parameters(), "lr": 5e-6},
    
#     # Projection layers (important for alignment)
#     {"params": model.visual_projection.parameters(), "lr": 5e-6},
#     {"params": model.text_projection.parameters(), "lr": 5e-6},
    
#     # Learned temperature
#     {"params": [logit_scale], "lr": 5e-6}
# ])


## Validation

In [ ]:
# val_pairs = []

# for item in val_dataset:
#     image = item["image"]
#     captions = item["caption"]
    
#     for caption in captions:
#         val_pairs.append((image, caption))

# len(val_pairs)

15505

In [25]:
# ==========================================
# CONTROLLED TRAIN DATA (40K WITH 25K UNIQUE IMAGES)
# ==========================================

from collections import defaultdict
import random

# Step 1: Map each image to its captions
unique_val_pairs = defaultdict(list)

for item in val_dataset:
    image = item["image"]
    captions = item["caption"]
    
    for caption in captions:
        unique_val_pairs[id(image)].append((image, caption))

print("Unique images:", len(unique_val_pairs))

Unique images: 3101


### Creates the Validation DataLoader
This cell wraps `val_pairs` in a `FlickrDataset` and loads it into a `DataLoader` with batch size 32, no shuffling, and the custom `collate_fn`. This loader is used for computing validation loss and recall during training.

In [26]:
# Step 2: Select 1 caption per unique image (25K pairs)

val_pairs = []

for image_id in unique_val_pairs:
    # randomly choose 1 caption per image
    pair = random.choice(unique_val_pairs[image_id])
    val_pairs.append(pair)

print("Unique image pairs:", len(val_pairs))

Unique image pairs: 3101


In [ ]:
# import random

# val_pairs = random.sample(val_pairs, 2000)
# # val_pairs = val_pairs[:2000]
# len(val_pairs)

2000

In [27]:
val_data = FlickrDataset(val_pairs)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

### Defines the `compute_validation_loss` Function
This function evaluates the model on the validation set without updating weights. For each batch, it computes normalized image and text embeddings, scales the similarity matrix by the learned temperature (`logit_scale.exp()`), and calculates the symmetric contrastive loss. It returns the average validation loss across all batches.

In [28]:
def compute_validation_loss(model, val_loader, logit_scale):
    
    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for batch in val_loader:
            
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            
            image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
            text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
            
            similarity = logit_scale.exp() * (image_embeds @ text_embeds.T)
            
            labels = torch.arange(similarity.size(0)).to(device)
            
            loss_i = torch.nn.functional.cross_entropy(similarity, labels)
            loss_t = torch.nn.functional.cross_entropy(similarity.T, labels)
            
            loss = (loss_i + loss_t) / 2
            
            total_val_loss += loss.item()
    
    return total_val_loss / len(val_loader)

### Defines the `compute_recall` Function
This function computes **Recall@10** on a given data loader. It extracts normalized image and text embeddings for the entire dataset, builds a pairwise cosine similarity matrix, and checks whether the correct image appears in the top 10 retrieved results for each text query. The ratio of correct retrievals is returned as the recall score.

In [29]:
def compute_recall(model, loader):
    
    model.eval()
    
    all_image_embeds = []
    all_text_embeds = []
    
    with torch.no_grad():
        for batch in loader:
            
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(**batch)
            
            image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
            text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
            
            all_image_embeds.append(image_embeds)
            all_text_embeds.append(text_embeds)
    
    all_image_embeds = torch.cat(all_image_embeds)
    all_text_embeds = torch.cat(all_text_embeds)
    
    similarity = all_text_embeds @ all_image_embeds.T
    similarity_np = similarity.cpu().numpy()
    
    recall_at_10 = 0
    
    for i in range(len(similarity_np)):
        ranked_indices = np.argsort(-similarity_np[i])
        if i in ranked_indices[:10]:
            recall_at_10 += 1
    
    return recall_at_10 / len(similarity_np)

### Initializes Early Stopping State
This cell sets up the variables for the early stopping strategy: `best_val_recall` tracks the best validation recall seen so far, `patience` defines how many epochs to wait without improvement before stopping, `patience_counter` counts consecutive non-improving epochs, and `best_model_state` stores the weights of the best model checkpoint.

In [31]:
best_val_recall = 0
patience = 2   # number of epochs to wait
patience_counter = 0

best_model_state = None

### Main Training Loop with Early Stopping
This cell runs the full fine-tuning loop for up to 5 epochs. In each epoch, the model processes every training batch, computes the symmetric contrastive loss scaled by the learnable temperature, and updates all parameters via backpropagation. After each epoch, validation loss and **Recall@10** are computed. If recall improves, the model weights are saved as the best checkpoint; otherwise, a patience counter increments. Training halts early if no improvement is seen for 2 consecutive epochs, and the best model is restored at the end.

In [32]:
epochs = 5

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch in train_loader:
        
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        
        image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
        text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
        
        similarity = logit_scale.exp() * (image_embeds @ text_embeds.T)
        
        labels = torch.arange(similarity.size(0)).to(device)
        
        loss_i = torch.nn.functional.cross_entropy(similarity, labels)
        loss_t = torch.nn.functional.cross_entropy(similarity.T, labels)
        loss = (loss_i + loss_t) / 2
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
           
    avg_train_loss = total_loss / num_batches
    
    # validation loss
    avg_val_loss = compute_validation_loss(model, val_loader, logit_scale)
    # avg_val_loss = val_loss / len(val_loader)
    
    # validation recall
    val_recall = compute_recall(model, val_loader)
    
    print(f"\nEpoch {epoch+1}")
    print("Average Train Loss:", avg_train_loss)
    print("Average Validation Loss:", avg_val_loss)
    print("Validation Recall@10:", val_recall)
    print("Temperature:", 1 / logit_scale.exp().item())

    # ----- EARLY STOPPING LOGIC -----
    if val_recall > best_val_recall:
        best_val_recall = val_recall
        patience_counter = 0
        
        # Save best model weights
        best_model_state = model.state_dict()
        print("New best model saved.")
        
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print("\nEarly stopping triggered.")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Best model restored.")


Epoch 1
Average Train Loss: 0.27902471858263017
Average Validation Loss: 0.21677308698597642
Validation Recall@10: 0.8819735569171235
Temperature: 0.06988835403160591
New best model saved.

Epoch 2
Average Train Loss: 0.12339569058716297
Average Validation Loss: 0.21014619050259442
Validation Recall@10: 0.8684295388584328
Temperature: 0.06976904957887765
No improvement. Patience: 1/2

Epoch 3
Average Train Loss: 0.0831636209309101
Average Validation Loss: 0.20151312303604538
Validation Recall@10: 0.8684295388584328
Temperature: 0.06963904551280774
No improvement. Patience: 2/2

Early stopping triggered.
Best model restored.


In [34]:
torch.save(model.state_dict(), "clip_learned_temp.pt")

In [35]:
len(val_pairs)

3101

### Recreates the Validation DataLoader
This cell rebuilds the validation `DataLoader` to ensure it reflects the current `val_pairs` list, especially after any modifications or re-sampling done in earlier cells.

In [36]:
val_data = FlickrDataset(val_pairs)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

### Sets Model to Evaluation Mode
This cell calls `model.eval()` to disable dropout and batch normalization updates, preparing the fine-tuned model for inference on the validation set.

In [37]:
model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

### Extracts All Validation Embeddings
This cell runs a no-gradient forward pass over the entire validation loader, collecting normalized image and text embeddings batch by batch. The embeddings are concatenated into `all_image_embeds` and `all_text_embeds` tensors, and their shapes are printed to confirm correctness.

In [38]:
all_image_embeds = []
all_text_embeds = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        
        image_embeds = torch.nn.functional.normalize(outputs.image_embeds, dim=1)
        text_embeds = torch.nn.functional.normalize(outputs.text_embeds, dim=1)
        
        all_image_embeds.append(image_embeds)
        all_text_embeds.append(text_embeds)

all_image_embeds = torch.cat(all_image_embeds)
all_text_embeds = torch.cat(all_text_embeds)

print(all_image_embeds.shape)
print(all_text_embeds.shape)

torch.Size([3101, 512])
torch.Size([3101, 512])


In [39]:
similarity = all_text_embeds @ all_image_embeds.T
temperature = 0.07
similarity = similarity / temperature

### Computes Recall@1, @5, and @10 on Validation Set
This cell evaluates the fine-tuned model's retrieval performance on the validation set. For each text query, it ranks all images by similarity and checks if the correct image appears in the top 1, 5, or 10 results. The recall scores are printed as the final validation performance metrics.

In [40]:
import numpy as np

similarity_np = similarity.cpu().numpy()

recall_at_1 = 0
recall_at_5 = 0
recall_at_10 = 0

for i in range(len(similarity_np)):
    
    # sort indices in descending similarity
    ranked_indices = np.argsort(-similarity_np[i])
    
    if i in ranked_indices[:1]:
        recall_at_1 += 1
        
    if i in ranked_indices[:5]:
        recall_at_5 += 1
        
    if i in ranked_indices[:10]:
        recall_at_10 += 1

N = len(similarity_np)

print("Recall@1:", recall_at_1 / N)
print("Recall@5:", recall_at_5 / N)
print("Recall@10:", recall_at_10 / N)

Recall@1: 0.5424056755885198
Recall@5: 0.8055465978716543
Recall@10: 0.8684295388584328
